In [1]:
from pathlib import Path
import os
import numpy as np
import torch
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

from transformers import pipeline

from src.shared.vkitti._loader import VKITTI2Loader

In [2]:
device = 0 if torch.cuda.is_available() else -1

print(f'Используем: {"GPU" if device == 0 else "CPU"}')

Используем: GPU


In [3]:
pipe = pipeline(

    task="depth-estimation",

    model="depth-anything/Depth-Anything-V2-Small-hf",

    device=device

)

print('✅ Depth Anything v2 загружен')

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

d:\sadekov\3dcv-project\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\elena\.cache\huggingface\hub\models--depth-anything--Depth-Anything-V2-Small-hf. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/99.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

✅ Depth Anything v2 загружен


In [4]:
import os

def predict_depth(image_path, save_path=None):

    """

    Предсказать depth для одного изображения.

    Возвращает np.array (H, W) с относительной глубиной 0–1.

    """

    image = Image.open(image_path).convert('RGB')

    result = pipe(image)

    depth = np.array(result['depth'])  # (H, W), относительная

    

    # Нормализуем в 0–1

    depth_normalized = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

    

    if save_path:

        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        np.save(save_path, depth_normalized.astype(np.float32))

    

    return depth_normalized

In [7]:
from src.shared.vkitti._loader import VKITTI2Loader

In [8]:
from pathlib import Path

DATA_ROOT = Path("data/vkitti2")

loader = VKITTI2Loader(DATA_ROOT)

frames = loader.list_frames("Scene01", "clone")

print("Кадров найдено:", len(frames))
print(frames[:5])

Кадров найдено: 447
['00000', '00001', '00002', '00003', '00004']


In [ ]:
scene = "Scene01"
variation = "clone"

image_path ="data" / scene / variation / "frames" / "rgb" / "Camera_0" / f"rgb_{frame_id}.jpg" 

In [24]:
from pathlib import Path

scene = "Scene01"
variation = "clone"
frame_id = frames[0]

image_path = f"./data/vkitti2/{scene}/{variation}/frames/rgb/Camera_0/rgb_{frame_id}.jpg"

save_path = f"./results/track_b/depth_pred_vkitti/scene/{frame_id}.npy"


depth_pred = predict_depth(image_path, save_path)

print(depth_pred.shape)
print(depth_pred.min(), depth_pred.max())
print("saved:", Path(save_path).exists())

(375, 1242)
0.0 0.9999999999607843
saved: True


In [28]:
from tqdm.notebook import tqdm

VKITTI_RGB = Path("data/vkitti2")

DEPTH_PRED_DIR = f"./results/track_b/depth_pred_vkitti"

loader = VKITTI2Loader(VKITTI_RGB)

for scene in loader.SCENES:

    frames = loader.list_frames(scene, 'clone')

    print(f'n{scene}: {len(frames)} кадров')
    for frame_id in tqdm(frames, desc=scene):

        rgb_path = f'{VKITTI_RGB}/{scene}/clone/frames/rgb/Camera_0/rgb_{frame_id}.jpg'

        save_path = f'{DEPTH_PRED_DIR}/{scene}/Camera_0/{frame_id}.npy'

        if not os.path.exists(save_path):

            predict_depth(rgb_path, save_path)

print('n✅ Инференс на VKITTI2 завершён')

nScene01: 447 кадров


Scene01:   0%|          | 0/447 [00:00<?, ?it/s]

nScene02: 233 кадров


Scene02:   0%|          | 0/233 [00:01<?, ?it/s]

nScene06: 270 кадров


...

nScene18: 339 кадров


Scene18:   0%|          | 0/339 [00:00<?, ?it/s]

nScene20: 837 кадров


Scene20:   0%|          | 0/837 [00:00<?, ?it/s]

n✅ Инференс на VKITTI2 завершён


In [ ]:
from tqdm.notebook import tqdm

VKITTI_RGB = Path("data/vkitti2")

DEPTH_PRED_DIR = f"./results/track_b/depth_pred_vkitti"

loader = VKITTI2Loader(VKITTI_RGB)

for scene in loader.SCENES:

    frames = loader.list_frames(scene, 'clone')

    print(f'n{scene}: {len(frames)} кадров')
    for frame_id in tqdm(frames, desc=scene):

        rgb_path = f'{VKITTI_RGB}/{scene}/clone/frames/rgb/Camera_1/rgb_{frame_id}.jpg'

        save_path = f'{DEPTH_PRED_DIR}/{scene}/Camera_1/{frame_id}.npy'

        if not os.path.exists(save_path):

            predict_depth(rgb_path, save_path)

print('n✅ Инференс на VKITTI2 завершён')

nScene01: 447 кадров


Scene01:   0%|          | 0/447 [00:00<?, ?it/s]

nScene02: 233 кадров


Scene02:   0%|          | 0/233 [00:00<?, ?it/s]

nScene06: 270 кадров


...

nScene18: 339 кадров


Scene18:   0%|          | 0/339 [00:00<?, ?it/s]

nScene20: 837 кадров


Scene20:   0%|          | 0/837 [00:00<?, ?it/s]

n✅ Инференс на VKITTI2 завершён


In [35]:
import numpy as np

for i in range(5):
    path = f"./results/track_b/depth_pred_vkitti/Scene18/Camera_0/0000{i}.npy"
    arr = np.load(path)

    print(f"\nFile: {path}")
    print("shape:", arr.shape)
    print("dtype:", arr.dtype)
    print("ndim:", arr.ndim)
    print("min:", arr.min())
    print("max:", arr.max())


File: ./results/track_b/depth_pred_vkitti/Scene18/Camera_0/00000.npy
shape: (375, 1242)
dtype: float32
ndim: 2
min: 0.0
max: 1.0

File: ./results/track_b/depth_pred_vkitti/Scene18/Camera_0/00001.npy
shape: (375, 1242)
dtype: float32
ndim: 2
min: 0.0
max: 1.0

File: ./results/track_b/depth_pred_vkitti/Scene18/Camera_0/00002.npy
shape: (375, 1242)
dtype: float32
ndim: 2
min: 0.0
max: 1.0

File: ./results/track_b/depth_pred_vkitti/Scene18/Camera_0/00003.npy
shape: (375, 1242)
dtype: float32
ndim: 2
min: 0.0
max: 1.0

File: ./results/track_b/depth_pred_vkitti/Scene18/Camera_0/00004.npy
shape: (375, 1242)
dtype: float32
ndim: 2
min: 0.0
max: 1.0
